# Objectives

### English

The objective of this notebook is to build the first current statistical representation of the 48 national teams qualified for the 2026 FIFA World Cup.

The pipeline combines the validated Current Roster Matching V02 with the International Player Database V02 in order to:

- Retrieve the historical statistical vector of each matched current player.
- Aggregate player-level statistics into one vector per national team.
- Measure the number of represented players and the effective squad coverage for each country.
- Validate the integrity, consistency, and completeness of the resulting team vectors.
- Export the Current Team Vector V01 dataset for use in the current match vector and prediction pipelines.

### Español

El objetivo de esta notebook es construir la primera representación estadística actual de las 48 selecciones clasificadas al Mundial de la FIFA 2026.

El pipeline combina el Current Roster Matching V02 validado con la International Player Database V02 para:

- Recuperar el vector estadístico histórico de cada jugador actual correctamente identificado.
- Agregar las estadísticas a nivel jugador en un único vector por selección nacional.
- Medir la cantidad de jugadores representados y la cobertura efectiva del plantel de cada país.
- Validar la integridad, consistencia y completitud de los vectores resultantes.
- Exportar el dataset Current Team Vector V01 para utilizarlo en los pipelines de current match vectors y predicción.

# Configuration

## Imports

In [27]:
import pandas as pd
import numpy as np
from pathlib import Path

## Paths

In [28]:
# Project directories
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Input files
CURRENT_ROSTER_MATCHING_PATH = (
    PROCESSED_DIR / "current_roster_matching_v02.csv"
)

PLAYER_DATABASE_PATH = (
    PROCESSED_DIR / "international_player_database_v02.csv"
)

# Output files
CURRENT_TEAM_VECTORS_PATH = (
    PROCESSED_DIR / "current_team_vectors_v01.csv"
)

TEAM_COVERAGE_PATH = (
    PROCESSED_DIR / "team_vector_coverage_v01.csv"
)

## Utility Functions

In [41]:
def retrieve_current_player_vectors(
    matched_rosters: pd.DataFrame,
    player_database: pd.DataFrame,
) -> pd.DataFrame:
    """
    Retrieve historical player vectors for validated current-roster matches.

    Parameters
    ----------
    matched_rosters : pd.DataFrame
        Current roster matching dataset containing validated player IDs.

    player_database : pd.DataFrame
        Consolidated historical player database containing one vector
        per unique player ID.

    Returns
    -------
    pd.DataFrame
        Current matched players enriched with their historical vectors.
    """

    roster_columns = [
        "country",
        "country_clean",
        "roster_name",
        "position",
        "shirt_number",
        "player_id",
        "database_name",
        "database_team",
        "match_method",
    ]

    database_columns = [
        "player_id",
        "player_name",
        "team_name",
        "matches_played",
        "competitions_played",
        "seasons_played",
        *stat_columns,
    ]

    current_player_vectors = (
        matched_rosters[roster_columns]
        .merge(
            player_database[database_columns],
            on="player_id",
            how="left",
            validate="one_to_one",
        )
        .rename(
            columns={
                "player_name": "historical_player_name",
                "team_name": "historical_team_name",
            }
        )
    )

    return current_player_vectors

In [44]:
def aggregate_current_team_vectors(
    current_player_vectors: pd.DataFrame,
    stat_columns: list[str],
) -> pd.DataFrame:
    """
    Aggregate current player historical vectors into one vector per national team.

    Parameters
    ----------
    current_player_vectors : pd.DataFrame
        Matched current players enriched with historical statistical vectors.

    stat_columns : list[str]
        Statistical columns to aggregate.

    Returns
    -------
    pd.DataFrame
        One summed statistical vector per represented national team.
    """

    current_team_vectors = (
        current_player_vectors
        .groupby("country_clean", as_index=False)[stat_columns]
        .sum()
    )

    return current_team_vectors

In [46]:
def build_team_coverage_report(
    roster_matching: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build current squad coverage metrics for every national team.

    Parameters
    ----------
    roster_matching : pd.DataFrame
        Full Current Roster Matching dataset, including matched and unmatched players.

    Returns
    -------
    pd.DataFrame
        Coverage report with squad size, matched players, unmatched players,
        and coverage ratio for each national team.
    """

    team_coverage = (
        roster_matching
        .groupby("country_clean", as_index=False)
        .agg(
            squad_size=("roster_name", "size"),
            matched_players=("matched", "sum"),
        )
    )

    team_coverage["matched_players"] = (
        team_coverage["matched_players"]
        .astype(int)
    )

    team_coverage["unmatched_players"] = (
        team_coverage["squad_size"]
        - team_coverage["matched_players"]
    )

    team_coverage["coverage"] = (
        team_coverage["matched_players"]
        / team_coverage["squad_size"]
    )

    return team_coverage

## Load Data

The two main inputs are loaded:

- **Current Roster Matching V02**, containing the validated correspondence between the current World Cup squads and the historical player database.
- **International Player Database V02**, containing the consolidated historical statistical vectors of international and club players.

In [29]:
current_roster_matching = pd.read_csv(
    CURRENT_ROSTER_MATCHING_PATH
)

player_database = pd.read_csv(
    PLAYER_DATABASE_PATH
)

print(
    "Current Roster Matching V02:",
    current_roster_matching.shape
)

print(
    "International Player Database V02:",
    player_database.shape
)

Current Roster Matching V02: (1248, 12)
International Player Database V02: (4058, 33)


### Initial Preview

In [30]:
print("\ncurrent_roster_matching")
display(current_roster_matching.head())

print("\nplayer_database")
display(player_database.head())


current_roster_matching


,country,country_clean,roster_name,position,shirt_number,player_id,database_name,database_team,matched,match_method,review_confidence,review_notes
0,Algeria (ALG),Algeria,MASTIL Melvin,Goalkeeper,1,NaN,NaN,NaN,False,unmatched,NaN,NaN
1,Algeria (ALG),Algeria,MANDI Aissa,Defender,2,6648.0,Aïssa Mandi,Algeria,True,token_subset_country,NaN,NaN
2,Algeria (ALG),Algeria,ABADA Achref,Defender,3,NaN,NaN,NaN,False,unmatched,NaN,NaN
3,Algeria (ALG),Algeria,TOUGAI Mohamed,Defender,4,160943.0,Mohamed Amine Tougai,Algeria,True,token_subset_country,NaN,NaN
4,Algeria (ALG),Algeria,BELAID Zineddine,Defender,5,NaN,NaN,NaN,False,unmatched,NaN,NaN



player_database


,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,...,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
0,2935,Nordi Mukiele Mulere,Paris Saint-Germain,0.0,0.0,0.0,3.0,0.0,399,26,...,2.0,2.0,107,4.0,5,0.0,0.0,0.0,0.0,0.0
1,2936,Christophe Kerbrat,Guingamp,0.0,0.0,0.0,0.0,0.0,12,3,...,0.0,0.0,18,0.0,0,0.0,0.0,0.0,0.0,0.0
2,2937,Christ-Emmanuel Faitout Maouassa,Montpellier,0.0,0.0,0.0,0.0,0.0,50,5,...,0.0,0.0,27,0.0,0,0.0,0.0,0.0,0.0,0.0
3,2940,Jean Eudès Aholou,Strasbourg,0.0,0.0,0.0,0.0,0.0,9,1,...,0.0,0.0,4,0.0,0,0.0,0.0,0.0,0.0,0.0
4,2941,Ismaïla Sarr,Senegal,11.0,2.0,3.0,5.0,0.0,547,39,...,2.0,2.0,250,0.0,30,0.0,1.0,0.0,0.0,0.0


### Initial Dataset Checks

In [31]:
assert not current_roster_matching.empty, (
    "Current Roster Matching V02 is empty."
)

assert not player_database.empty, (
    "International Player Database V02 is empty."
)

assert "country" in current_roster_matching.columns, (
    "Missing 'country' column in Current Roster Matching V02."
)

assert "roster_name" in current_roster_matching.columns, (
    "Missing 'roster_name' column in Current Roster Matching V02."
)

assert "player_id" in current_roster_matching.columns, (
    "Missing 'player_id' column in Current Roster Matching V02."
)

assert "matched" in current_roster_matching.columns, (
    "Missing 'matched' column in Current Roster Matching V02."
)

assert "player_id" in player_database.columns, (
    "Missing 'player_id' column in International Player Database V02."
)

assert "player_name" in player_database.columns, (
    "Missing 'player_name' column in International Player Database V02."
)

print("Initial dataset checks passed.")

Initial dataset checks passed.


In [32]:
assert (
    current_roster_matching
    .loc[current_roster_matching["matched"], "player_id"]
    .is_unique
), "Matched player IDs must be unique."

In [33]:
duplicate_player_ids = (
    current_roster_matching
    .loc[current_roster_matching["matched"]]
    .groupby("player_id")
    .size()
)

n_duplicates = (duplicate_player_ids > 1).sum()

print(f"Duplicate matched player IDs: {n_duplicates}")

assert n_duplicates == 0, (
    "Matched player IDs must be unique."
)

print("Matched player IDs are unique.")

Duplicate matched player IDs: 0
Matched player IDs are unique.


### Duplicate Player Integrity Check

A duplicated historical player identifier was initially detected in the Brazilian roster. The inconsistency was resolved upstream in Notebook 16 by retaining the exact normalized match and marking the duplicated roster entry as unmatched.

The current input is therefore expected to contain no duplicated matched player IDs.

# Build Current Team Vector

## Retrieve Historical Player Vectors

- Only validated roster matches are retained.

- Each matched current player is linked to the International Player Database V02 through the unique `player_id`. This operation retrieves the consolidated historical statistical vector that will later be aggregated at national-team level.

- The merge is validated as one-to-one to prevent duplicated player vectors from entering the current team representation.

### Prepare player identifiers


In [37]:
current_roster_matching["player_id"] = pd.to_numeric(
    current_roster_matching["player_id"],
    errors="coerce",
).astype("Int64")

player_database["player_id"] = pd.to_numeric(
    player_database["player_id"],
    errors="raise",
).astype("Int64")

In [38]:
assert player_database["player_id"].notna().all(), (
    "Player Database contains missing player IDs."
)

assert player_database["player_id"].is_unique, (
    "Player IDs must be unique in International Player Database V02."
)

print("Player Database IDs are valid and unique.")

Player Database IDs are valid and unique.


### Select matched current players

In [39]:
matched_current_players = (
    current_roster_matching
    .loc[current_roster_matching["matched"]]
    .copy()
    .reset_index(drop=True)
)

print("Matched current players:", len(matched_current_players))
print(
    "Countries represented:",
    matched_current_players["country_clean"].nunique(),
)

Matched current players: 630
Countries represented: 43


### Identify player statistical columns

In [40]:
PLAYER_METADATA_COLUMNS = [
    "player_id",
    "player_name",
    "team_name",
    "matches_played",
    "competitions_played",
    "seasons_played",
]

stat_columns = [
    column
    for column in player_database.columns
    if column not in PLAYER_METADATA_COLUMNS
]

print("Statistical columns:", len(stat_columns))
print(stat_columns)

Statistical columns: 27
['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Interception', 'Miscontrol', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot', 'Error', 'Offside', 'Own Goal Against', 'Camera On', 'Own Goal For']


In [42]:
current_player_vectors = retrieve_current_player_vectors(
    matched_rosters=matched_current_players,
    player_database=player_database,
)

print("Current player vectors:", current_player_vectors.shape)

display(current_player_vectors.head())

Current player vectors: (630, 41)


,country,country_clean,roster_name,position,shirt_number,player_id,database_name,database_team,match_method,historical_player_name,...,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
0,Algeria (ALG),Algeria,MANDI Aissa,Defender,2,6648,Aïssa Mandi,Algeria,token_subset_country,Aïssa Mandi,...,0.0,0.0,68,0.0,4,0.0,1.0,0.0,0.0,0.0
1,Algeria (ALG),Algeria,TOUGAI Mohamed,Defender,4,160943,Mohamed Amine Tougai,Algeria,token_subset_country,Mohamed Amine Tougai,...,0.0,0.0,6,1.0,0,0.0,0.0,0.0,0.0,0.0
2,Algeria (ALG),Algeria,ZERROUKI Ramiz,Midfielder,6,38345,Ramiz Larbi Zerrouki,Algeria,token_subset_country,Ramiz Larbi Zerrouki,...,0.0,0.0,32,0.0,0,1.0,0.0,0.0,0.0,0.0
3,Algeria (ALG),Algeria,MAHREZ Riyad,Forward,7,3814,Riyad Mahrez,Algeria,token_subset_country,Riyad Mahrez,...,0.0,0.0,18,0.0,1,0.0,0.0,0.0,0.0,0.0
4,Algeria (ALG),Algeria,AOUAR Houssem,Midfielder,8,4308,Houssem Aouar,Algeria,token_subset_country,Houssem Aouar,...,0.0,0.0,29,0.0,2,0.0,0.0,0.0,0.0,0.0


### Retrieval integrity checks

In [43]:
assert len(current_player_vectors) == len(matched_current_players), (
    "The player-vector merge changed the number of matched players."
)

assert current_player_vectors["player_id"].is_unique, (
    "Current player vectors contain duplicated player IDs."
)

assert current_player_vectors["historical_player_name"].notna().all(), (
    "Some matched players were not found in the Player Database."
)

assert current_player_vectors[stat_columns].notna().all().all(), (
    "Missing values were found in historical statistical vectors."
)

assert (
    current_player_vectors["country_clean"].nunique()
    == matched_current_players["country_clean"].nunique()
), "Country representation changed during the player-vector merge."

print("Historical player vectors retrieved successfully.")
print("All retrieval integrity checks passed.")

Historical player vectors retrieved successfully.
All retrieval integrity checks passed.


## Aggregate Current Team Vectors

The historical vectors of all matched current players are aggregated by national team.

Event statistics are summed to represent the total historical contribution available within each current squad. Coverage metadata is calculated separately so that the strength and reliability of every team vector can be interpreted together.

In [45]:
current_team_vectors = aggregate_current_team_vectors(
    current_player_vectors=current_player_vectors,
    stat_columns=stat_columns,
)

print("Current Team Vectors:", current_team_vectors.shape)

display(current_team_vectors.head())

Current Team Vectors: (43, 28)


,country_clean,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,Dispossessed,Dribble,...,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
0,Algeria,9.0,1.0,1753,135,56,1570,51,35,37,...,3.0,3.0,471,2.0,33,2.0,2.0,1.0,0.0,0.0
1,Argentina,91.0,28.0,26859,1799,591,23232,449,632,1170,...,18.0,18.0,4893,19.0,1022,11.0,5.0,2.0,0.0,0.0
2,Australia,8.0,1.0,1538,178,86,1206,74,55,71,...,1.0,1.0,538,2.0,23,0.0,0.0,1.0,0.0,0.0
3,Austria,43.0,4.0,4846,508,222,4088,156,126,122,...,3.0,3.0,1916,6.0,139,3.0,1.0,0.0,0.0,0.0
4,Belgium,23.0,1.0,5399,509,159,4688,121,120,192,...,2.0,2.0,1477,5.0,142,4.0,1.0,0.0,0.0,0.0


## Build Team Coverage Report

In [47]:
team_vector_coverage = build_team_coverage_report(
    roster_matching=current_roster_matching,
)

print("Team Coverage Report:", team_vector_coverage.shape)

display(
    team_vector_coverage
    .sort_values(
        ["coverage", "country_clean"],
        ascending=[True, True],
    )
)

Team Coverage Report: (48, 5)


,country_clean,squad_size,matched_players,unmatched_players,coverage
12,Curaçao,26,0,26,0.000000
23,Iraq,26,0,26,0.000000
25,Jordan,26,0,26,0.000000
30,New Zealand,26,0,26,0.000000
47,Uzbekistan,26,0,26,0.000000
5,Bosnia And Herzegovina,26,2,24,0.076923
31,Norway,26,2,24,0.076923
21,Haiti,26,3,23,0.115385
41,Sweden,26,4,22,0.153846
2,Australia,26,8,18,0.307692


## Combine Vectors and Coverage

In [48]:
current_team_vectors_complete = (
    team_vector_coverage
    .merge(
        current_team_vectors,
        on="country_clean",
        how="left",
        validate="one_to_one",
    )
)

current_team_vectors_complete[stat_columns] = (
    current_team_vectors_complete[stat_columns]
    .fillna(0)
)

print(
    "Complete Current Team Vectors:",
    current_team_vectors_complete.shape,
)

display(
    current_team_vectors_complete
    .sort_values("coverage")
    .head(10)
)

Complete Current Team Vectors: (48, 32)


,country_clean,squad_size,matched_players,unmatched_players,coverage,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,...,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
12,Curaçao,26,0,26,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30,New Zealand,26,0,26,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25,Jordan,26,0,26,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,Iraq,26,0,26,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
47,Uzbekistan,26,0,26,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31,Norway,26,2,24,0.076923,1.0,0.0,65.0,15.0,7.0,...,0.0,0.0,36.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
5,Bosnia And Herzegovina,26,2,24,0.076923,3.0,0.0,802.0,96.0,42.0,...,0.0,0.0,388.0,5.0,19.0,0.0,1.0,0.0,0.0,0.0
21,Haiti,26,3,23,0.115385,1.0,0.0,214.0,16.0,9.0,...,0.0,0.0,90.0,0.0,5.0,1.0,0.0,0.0,0.0,0.0
41,Sweden,26,4,22,0.153846,0.0,0.0,612.0,59.0,22.0,...,0.0,0.0,178.0,2.0,25.0,0.0,0.0,0.0,0.0,0.0
2,Australia,26,8,18,0.307692,8.0,1.0,1538.0,178.0,86.0,...,1.0,1.0,538.0,2.0,23.0,0.0,0.0,1.0,0.0,0.0


## Aggregation integrity checks

In [49]:
assert current_team_vectors["country_clean"].is_unique, (
    "Represented team vectors contain duplicated countries."
)

assert len(current_team_vectors) == 43, (
    "Expected 43 represented national teams."
)

assert len(team_vector_coverage) == 48, (
    "Coverage report must contain all 48 national teams."
)

assert team_vector_coverage["country_clean"].is_unique, (
    "Coverage report contains duplicated countries."
)

assert team_vector_coverage["squad_size"].eq(26).all(), (
    "Every current squad must contain exactly 26 players."
)

assert team_vector_coverage["matched_players"].sum() == 630, (
    "Coverage report does not preserve the 630 matched players."
)

assert team_vector_coverage["coverage"].between(0, 1).all(), (
    "Coverage values must remain between 0 and 1."
)

assert len(current_team_vectors_complete) == 48, (
    "Complete team vectors must contain all 48 national teams."
)

assert current_team_vectors_complete["country_clean"].is_unique, (
    "Complete team vectors contain duplicated countries."
)

assert current_team_vectors_complete[stat_columns].notna().all().all(), (
    "Complete team vectors contain missing statistical values."
)

assert (
    current_team_vectors_complete[stat_columns] >= 0
).all().all(), (
    "Negative values were found in aggregated statistics."
)

assert (
    current_team_vectors_complete[stat_columns].sum().sum()
    == current_player_vectors[stat_columns].sum().sum()
), "Aggregated team totals do not match player-vector totals."

print("Current Team Vectors aggregated successfully.")
print("All aggregation integrity checks passed.")

Current Team Vectors aggregated successfully.
All aggregation integrity checks passed.


## Coverage Analysis

- The coverage report quantifies how well each current national team is represented by the historical player database.

- Coverage should always be interpreted together with the aggregated team vectors, since lower coverage indicates that a larger proportion of the current squad could not be represented using historical data.

## Coverage summary

In [50]:
coverage_summary = pd.DataFrame({
    "Metric": [
        "National teams",
        "Represented teams",
        "Players in current rosters",
        "Matched players",
        "Average coverage",
        "Median coverage",
        "Teams with full coverage",
        "Teams with >75% coverage",
        "Teams with >50% coverage",
        "Teams with zero coverage",
    ],
    "Value": [
        len(team_vector_coverage),
        (team_vector_coverage["matched_players"] > 0).sum(),
        team_vector_coverage["squad_size"].sum(),
        team_vector_coverage["matched_players"].sum(),
        round(team_vector_coverage["coverage"].mean(), 3),
        round(team_vector_coverage["coverage"].median(), 3),
        (team_vector_coverage["coverage"] == 1).sum(),
        (team_vector_coverage["coverage"] >= 0.75).sum(),
        (team_vector_coverage["coverage"] >= 0.50).sum(),
        (team_vector_coverage["coverage"] == 0).sum(),
    ]
})

coverage_summary

,Metric,Value
0,National teams,48.000
1,Represented teams,43.000
2,Players in current rosters,1248.000
3,Matched players,630.000
4,Average coverage,0.505
5,Median coverage,0.577
6,Teams with full coverage,0.000
7,Teams with >75% coverage,9.000
8,Teams with >50% coverage,27.000
9,Teams with zero coverage,5.000


### Highest Coverage

In [51]:
display(
    team_vector_coverage
    .sort_values(
        "coverage",
        ascending=False,
    )
    .head(10)
)

,country_clean,squad_size,matched_players,unmatched_players,coverage
34,Portugal,26,22,4,0.846154
44,Türkiye,26,22,4,0.846154
18,France,26,22,4,0.846154
19,Germany,26,22,4,0.846154
40,Spain,26,21,5,0.807692
1,Argentina,26,20,6,0.769231
9,Colombia,26,20,6,0.769231
3,Austria,26,20,6,0.769231
42,Switzerland,26,20,6,0.769231
38,Senegal,26,19,7,0.730769


### Lowest Coverage

In [52]:
display(
    team_vector_coverage
    .sort_values(
        "coverage",
        ascending=True,
    )
    .head(10)
)

,country_clean,squad_size,matched_players,unmatched_players,coverage
12,Curaçao,26,0,26,0.000000
30,New Zealand,26,0,26,0.000000
25,Jordan,26,0,26,0.000000
23,Iraq,26,0,26,0.000000
47,Uzbekistan,26,0,26,0.000000
31,Norway,26,2,24,0.076923
5,Bosnia And Herzegovina,26,2,24,0.076923
21,Haiti,26,3,23,0.115385
41,Sweden,26,4,22,0.153846
2,Australia,26,8,18,0.307692


## Current Team Coverage Ranking

Coverage ranking for the 48 qualified national teams based on the proportion of players successfully matched to the historical player database.

In [57]:
coverage_ranking = (
    team_vector_coverage
    .copy()
    .assign(
        coverage_percent=lambda df: (
            df["coverage"] * 100
        ).round(1)
    )
    .drop(columns=["squad_size", "coverage"])
    .rename(
        columns={
            "country_clean": "Country",
            "matched_players": "Matched",
            "unmatched_players": "Unmatched",
            "coverage_percent": "Coverage (%)",
        }
    )
    .sort_values(
        ["Coverage (%)", "Matched", "Country"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

display(coverage_ranking)

,Country,Matched,Unmatched,Coverage (%)
0,France,22,4,84.6
1,Germany,22,4,84.6
2,Portugal,22,4,84.6
3,Türkiye,22,4,84.6
4,Spain,21,5,80.8
5,Argentina,20,6,76.9
6,Austria,20,6,76.9
7,Colombia,20,6,76.9
8,Switzerland,20,6,76.9
9,Canada,19,7,73.1


# Export Outputs

In [53]:
current_team_vectors_complete.to_csv(
    CURRENT_TEAM_VECTORS_PATH,
    index=False,
)

team_vector_coverage.to_csv(
    TEAM_COVERAGE_PATH,
    index=False,
)

print(f"Current Team Vectors exported to: {CURRENT_TEAM_VECTORS_PATH}")
print(f"Coverage report exported to: {TEAM_COVERAGE_PATH}")

Current Team Vectors exported to: ..\data\processed\current_team_vectors_v01.csv
Coverage report exported to: ..\data\processed\team_vector_coverage_v01.csv


## Checks

In [54]:
assert CURRENT_TEAM_VECTORS_PATH.exists(), (
    "Current Team Vectors export failed."
)

assert TEAM_COVERAGE_PATH.exists(), (
    "Coverage report export failed."
)

exported_vectors = pd.read_csv(CURRENT_TEAM_VECTORS_PATH)
exported_coverage = pd.read_csv(TEAM_COVERAGE_PATH)

assert exported_vectors.shape == current_team_vectors_complete.shape

assert exported_coverage.shape == team_vector_coverage.shape

print("Export checks passed.")
print("Notebook 17 completed successfully.")

Export checks passed.
Notebook 17 completed successfully.


# Discussion

### English

- The Current Team Vector V01 successfully aggregates the historical statistical profiles of the players currently representing each qualified national team.

- Coverage analysis shows that 43 of the 48 qualified teams have at least one historical player representation, while five teams remain completely unmatched due to insufficient historical data.

- The aggregated vectors and the coverage report provide complementary information. The statistical vectors summarize the historical activity of each squad, whereas the coverage metric quantifies the completeness - and reliability of that representation.

- These current team vectors constitute the first fully current dataset generated by the project and provide the direct input required to build current match vectors in the next stage of the pipeline.

### Español

- Current Team Vector V01 agrega correctamente los perfiles estadísticos históricos de los jugadores que actualmente integran cada una de las selecciones clasificadas.

- El análisis de cobertura muestra que 43 de las 48 selecciones poseen al menos un jugador representado en la base histórica, mientras que cinco equipos permanecen completamente sin representación debido a la falta de información histórica disponible.

- Los vectores agregados y el reporte de cobertura aportan información complementaria. Los vectores resumen la actividad histórica del plantel, mientras que la cobertura cuantifica el nivel de representación y la confiabilidad de cada selección.

- Estos vectores actuales constituyen el primer dataset completamente orientado al Mundial 2026 y representan la entrada directa para construir los Current Match Vectors en la siguiente etapa del pipeline.

# Conclusion

### English

- Historical player vectors were successfully retrieved for all validated current roster matches.
- Player-level statistics were aggregated into one statistical vector per national team.
- Coverage metrics were computed for all 48 qualified teams.
- Integrity checks confirmed the consistency of the aggregation process and the preservation of all statistical information.
- Current Team Vector V01 is ready to be used for the construction of current match vectors.

### Español

- Se recuperaron correctamente los vectores históricos de todos los jugadores validados en los planteles actuales.
- Las estadísticas individuales fueron agregadas en un único vector por selección nacional.
- Se calcularon métricas de cobertura para las 48 selecciones clasificadas.
- Los controles de integridad confirmaron la consistencia del proceso de agregación y la preservación de toda la información estadística.
- Current Team Vector V01 queda preparado para la construcción de los Current Match Vectors.